In [1]:
# ================================================================
# REAL INSTAGRAM IMAGE - RESNET-50 DEEP FEATURE EXTRACTION
# AI-Based Instagram Engagement Prediction and Content Optimization System
# ================================================================

import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models import resnet50, ResNet50_Weights

warnings.filterwarnings("ignore")


# ================================================================
# 1. PROJECT CONFIGURATION
# ================================================================

PROJECT_ROOT = Path(r"d:\newwwwwwww\AiBasedInstagramPrediction")

DATASET_1_DIR = PROJECT_ROOT / "datasets" / "raw" / "instagram_data"
DATASET_2_DIR = PROJECT_ROOT / "datasets" / "raw" / "instagram_data2"

CAPTION_1 = DATASET_1_DIR / "captions_csv.csv"
CAPTION_2 = DATASET_2_DIR / "captions_csv2.csv"

IMG_1_DIR = DATASET_1_DIR / "img"
IMG_2_DIR = DATASET_2_DIR / "img2"

OUTPUT_DIR = PROJECT_ROOT / "datasets" / "processed" / "real_image_features"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILE = OUTPUT_DIR / "real_resnet50_features.npy"
METADATA_FILE = OUTPUT_DIR / "real_resnet50_metadata.csv"
SUMMARY_FILE = OUTPUT_DIR / "real_resnet50_extraction_summary.json"


print("=" * 70)
print("REAL INSTAGRAM IMAGE - RESNET-50 FEATURE EXTRACTION")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nDataset 1:")
print(DATASET_1_DIR)

print("\nDataset 2:")
print(DATASET_2_DIR)

print("\nOutput directory:")
print(OUTPUT_DIR)


# ================================================================
# 2. VALIDATE DIRECTORIES
# ================================================================

required_paths = [
    DATASET_1_DIR,
    DATASET_2_DIR,
    IMG_1_DIR,
    IMG_2_DIR,
    CAPTION_1,
    CAPTION_2
]

missing_paths = [str(p) for p in required_paths if not p.exists()]

if missing_paths:
    print("\nMISSING PATHS:")
    for p in missing_paths:
        print(" -", p)

    raise FileNotFoundError(
        "One or more required dataset paths do not exist."
    )

print("\nPATH VALIDATION: PASSED")


# ================================================================
# 3. LOAD CAPTION METADATA
# ================================================================

print("\n" + "=" * 70)
print("LOADING REAL IMAGE METADATA")
print("=" * 70)


def load_dataset_1():
    df = pd.read_csv(CAPTION_1)

    print("\nDataset 1 columns:")
    print(list(df.columns))

    # Expected:
    # Sr No | Image File | Caption

    image_col = None
    caption_col = None

    for col in df.columns:
        name = str(col).strip().lower()

        if "image" in name:
            image_col = col

        if "caption" in name:
            caption_col = col

    if image_col is None:
        raise ValueError("Could not identify image column in dataset 1.")

    if caption_col is None:
        caption_col = None

    result = pd.DataFrame()

    result["record_id"] = np.arange(1, len(df) + 1)

    result["image_file"] = df[image_col].astype(str)

    if caption_col is not None:
        result["caption"] = df[caption_col].fillna("").astype(str)
    else:
        result["caption"] = ""

    result["dataset_source"] = "instagram_data"

    return result


def load_dataset_2():
    df = pd.read_csv(CAPTION_2)

    print("\nDataset 2 columns:")
    print(list(df.columns))

    # Dataset 2 has malformed-looking column names in the audit.
    # Detect image and caption fields robustly.

    image_col = None
    caption_col = None

    for col in df.columns:

        name = str(col).strip().lower()

        if (
            "img2/" in name
            or "image" in name
            or "insta" in name
        ):
            image_col = col

    # Caption is generally the remaining textual column.
    object_columns = [
        c for c in df.columns
        if df[c].dtype == "object"
    ]

    if object_columns:
        candidates = [
            c for c in object_columns
            if c != image_col
        ]

        if candidates:
            caption_col = candidates[-1]

    if image_col is None:
        # Fallback: second column
        if len(df.columns) >= 2:
            image_col = df.columns[1]

    if caption_col is None:
        # Fallback: third column
        if len(df.columns) >= 3:
            caption_col = df.columns[2]

    print("\nDetected dataset 2 image column:", image_col)
    print("Detected dataset 2 caption column:", caption_col)

    result = pd.DataFrame()

    start_id = 20516

    result["record_id"] = np.arange(
        start_id,
        start_id + len(df)
    )

    result["image_file"] = df[image_col].astype(str)

    if caption_col is not None:
        result["caption"] = df[caption_col].fillna("").astype(str)
    else:
        result["caption"] = ""

    result["dataset_source"] = "instagram_data2"

    return result


df1 = load_dataset_1()
df2 = load_dataset_2()

metadata = pd.concat(
    [df1, df2],
    ignore_index=True
)

print("\n" + "=" * 70)
print("COMBINED METADATA")
print("=" * 70)

print("Total records:", len(metadata))

print("\nDataset distribution:")
print(metadata["dataset_source"].value_counts())


# ================================================================
# 4. RESOLVE IMAGE PATHS
# ================================================================

print("\n" + "=" * 70)
print("RESOLVING IMAGE PATHS")
print("=" * 70)


def resolve_image_path(row):

    image_file = str(row["image_file"]).strip()

    source = row["dataset_source"]

    if source == "instagram_data":
        base_dir = IMG_1_DIR

    else:
        base_dir = IMG_2_DIR

    # Remove possible directory prefix
    image_file_clean = image_file.replace("\\", "/")

    image_file_clean = image_file_clean.replace(
        "img/",
        ""
    )

    image_file_clean = image_file_clean.replace(
        "img2/",
        ""
    )

    image_file_clean = image_file_clean.strip("/")

    candidate = base_dir / image_file_clean

    # Dataset entries may not contain .jpg
    if candidate.exists():
        return candidate

    extensions = [
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
        ".JPG",
        ".JPEG",
        ".PNG",
        ".WEBP"
    ]

    for ext in extensions:

        candidate_ext = base_dir / (
            image_file_clean + ext
        )

        if candidate_ext.exists():
            return candidate_ext

    return None


metadata["image_path"] = metadata.apply(
    resolve_image_path,
    axis=1
)

valid_mask = metadata["image_path"].notna()

print("\nTotal records :", len(metadata))
print("Valid images  :", valid_mask.sum())
print("Missing images:", (~valid_mask).sum())

match_rate = (
    valid_mask.sum() / len(metadata) * 100
)

print(f"Match rate    : {match_rate:.2f}%")

if match_rate < 99:
    raise ValueError(
        "Image match rate is unexpectedly low. "
        "Check dataset paths before continuing."
    )

metadata = metadata[valid_mask].reset_index(drop=True)


# ================================================================
# 5. DEVICE
# ================================================================

print("\n" + "=" * 70)
print("DEVICE CONFIGURATION")
print("=" * 70)

if torch.cuda.is_available():

    device = torch.device("cuda")

    print("CUDA available: YES")
    print("GPU:", torch.cuda.get_device_name(0))

else:

    device = torch.device("cpu")

    print("CUDA available: NO")
    print("Using CPU")

print("Device:", device)


# ================================================================
# 6. RESNET-50 MODEL
# ================================================================

print("\n" + "=" * 70)
print("LOADING PRETRAINED RESNET-50")
print("=" * 70)

weights = ResNet50_Weights.DEFAULT

model = resnet50(weights=weights)

# Remove final classification layer.
# Output becomes 2048-dimensional deep visual representation.

model.fc = nn.Identity()

model = model.to(device)

model.eval()

transform = weights.transforms()

print("Architecture : ResNet-50")
print("Feature size : 2048")
print("Pretrained   : ImageNet")
print("Final layer  : Removed")
print("Mode         : Evaluation")


# ================================================================
# 7. DATASET CLASS
# ================================================================

class InstagramImageDataset(Dataset):

    def __init__(self, dataframe, transform):

        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        path = row["image_path"]

        try:

            image = Image.open(path).convert("RGB")

            image = self.transform(image)

            return image, idx

        except (
            UnidentifiedImageError,
            OSError,
            ValueError
        ):

            # Return blank image if an individual image fails.
            blank = Image.new(
                "RGB",
                (224, 224),
                (0, 0, 0)
            )

            blank = self.transform(blank)

            return blank, idx


# ================================================================
# 8. DATALOADER
# ================================================================

BATCH_SIZE = 64 if device.type == "cuda" else 16

NUM_WORKERS = 0

dataset = InstagramImageDataset(
    metadata,
    transform
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

print("\n" + "=" * 70)
print("DATALOADER")
print("=" * 70)

print("Records     :", len(dataset))
print("Batch size  :", BATCH_SIZE)
print("Batches     :", len(loader))
print("Workers     :", NUM_WORKERS)


# ================================================================
# 9. FEATURE EXTRACTION
# ================================================================

print("\n" + "=" * 70)
print("EXTRACTING RESNET-50 DEEP FEATURES")
print("=" * 70)

print("\nExpected feature shape:")
print(
    f"({len(metadata)}, 2048)"
)

print("\nStarting extraction...")
print("This may take several minutes.")

start_time = time.time()

features = np.zeros(
    (len(metadata), 2048),
    dtype=np.float32
)

processed = 0

with torch.no_grad():

    for batch_idx, (images, indices) in enumerate(loader):

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        outputs = outputs.detach().cpu().numpy()

        features[indices.numpy()] = outputs

        processed += len(indices)

        if (
            batch_idx == 0
            or (batch_idx + 1) % 25 == 0
            or processed == len(metadata)
        ):

            elapsed = time.time() - start_time

            speed = (
                processed / elapsed
                if elapsed > 0
                else 0
            )

            percent = (
                processed / len(metadata) * 100
            )

            print(
                f"Progress: {processed:>6}/{len(metadata)} "
                f"({percent:6.2f}%) | "
                f"Speed: {speed:7.2f} images/sec"
            )


elapsed = time.time() - start_time

print("\n" + "=" * 70)
print("FEATURE EXTRACTION COMPLETED")
print("=" * 70)

print(
    f"Processed images : {processed}"
)

print(
    f"Feature shape    : {features.shape}"
)

print(
    f"Time taken       : {elapsed / 60:.2f} minutes"
)


# ================================================================
# 10. FEATURE QUALITY CHECK
# ================================================================

print("\n" + "=" * 70)
print("FEATURE QUALITY CHECK")
print("=" * 70)

print("Shape:", features.shape)

print(
    "NaN values:",
    np.isnan(features).sum()
)

print(
    "Infinite values:",
    np.isinf(features).sum()
)

print(
    "Minimum:",
    float(features.min())
)

print(
    "Maximum:",
    float(features.max())
)

print(
    "Mean:",
    float(features.mean())
)

print(
    "Std:",
    float(features.std())
)

if np.isnan(features).any():
    raise ValueError(
        "NaN values detected in extracted features."
    )

if np.isinf(features).any():
    raise ValueError(
        "Infinite values detected in extracted features."
    )

print("\nFEATURE QUALITY: PASSED")


# ================================================================
# 11. SAVE FEATURES
# ================================================================

print("\n" + "=" * 70)
print("SAVING RESNET-50 FEATURES")
print("=" * 70)

np.save(
    FEATURE_FILE,
    features
)

print("\nFeatures saved:")
print(FEATURE_FILE)


# ================================================================
# 12. SAVE METADATA
# ================================================================

metadata_to_save = metadata.copy()

metadata_to_save["feature_row"] = np.arange(
    len(metadata_to_save)
)

# Convert Path objects to strings
metadata_to_save["image_path"] = (
    metadata_to_save["image_path"]
    .astype(str)
)

metadata_to_save.to_csv(
    METADATA_FILE,
    index=False
)

print("\nMetadata saved:")
print(METADATA_FILE)


# ================================================================
# 13. SAVE EXTRACTION SUMMARY
# ================================================================

summary = {

    "project":
        "AI-Based Instagram Engagement Prediction and Content Optimization System",

    "feature_extractor":
        "ResNet-50",

    "pretrained_weights":
        "ImageNet",

    "feature_dimension":
        2048,

    "total_metadata_records":
        int(len(metadata)),

    "dataset_1_records":
        int(
            (metadata["dataset_source"] ==
             "instagram_data").sum()
        ),

    "dataset_2_records":
        int(
            (metadata["dataset_source"] ==
             "instagram_data2").sum()
        ),

    "valid_images":
        int(len(metadata)),

    "missing_images":
        0,

    "match_rate_percent":
        float(match_rate),

    "device":
        str(device),

    "batch_size":
        BATCH_SIZE,

    "extraction_time_seconds":
        float(elapsed),

    "feature_shape":
        list(features.shape),

    "nan_values":
        int(np.isnan(features).sum()),

    "infinite_values":
        int(np.isinf(features).sum())
}

with open(
    SUMMARY_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        summary,
        f,
        indent=4
    )

print("\nSummary saved:")
print(SUMMARY_FILE)


# ================================================================
# 14. FINAL VALIDATION
# ================================================================

print("\n" + "=" * 70)
print("FINAL RESNET-50 EXTRACTION VALIDATION")
print("=" * 70)

print(
    f"Real images processed : {len(metadata)}"
)

print(
    f"Deep feature dimension : {features.shape[1]}"
)

print(
    f"Feature matrix shape   : {features.shape}"
)

print(
    f"NaN values             : {np.isnan(features).sum()}"
)

print(
    f"Infinite values        : {np.isinf(features).sum()}"
)

print(
    f"Extraction time        : {elapsed / 60:.2f} minutes"
)

print("\nFiles created:")

print(
    "1.",
    FEATURE_FILE
)

print(
    "2.",
    METADATA_FILE
)

print(
    "3.",
    SUMMARY_FILE
)

print("\n" + "=" * 70)
print("REAL IMAGE DEEP FEATURE EXTRACTION COMPLETED")
print("=" * 70)

print(
    "\nREADY FOR REAL-DATA MULTIMODAL FEATURE ANALYSIS"
)

REAL INSTAGRAM IMAGE - RESNET-50 FEATURE EXTRACTION

Project root:
d:\newwwwwwww\AiBasedInstagramPrediction

Dataset 1:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data

Dataset 2:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data2

Output directory:
d:\newwwwwwww\AiBasedInstagramPrediction\datasets\processed\real_image_features

PATH VALIDATION: PASSED

LOADING REAL IMAGE METADATA

Dataset 1 columns:
['Sr No', 'Image File', 'Caption']

Dataset 2 columns:
['20516', 'img2/insta20516', 'wHaT dAy Is It Even #stayhomeclub']

Detected dataset 2 image column: img2/insta20516
Detected dataset 2 caption column: wHaT dAy Is It Even #stayhomeclub

COMBINED METADATA
Total records: 34926

Dataset distribution:
dataset_source
instagram_data     20515
instagram_data2    14411
Name: count, dtype: int64

RESOLVING IMAGE PATHS

Total records : 34926
Valid images  : 34926
Missing images: 0
Match rate    : 100.00%

DEVICE CONFIGURATION
CUDA available: NO
Using CPU
D